# 04 — Mapped CSV to Feature Store to a training dataset

This standalone Features walkthrough uses already-provisioned groups and the
deployment bindings described in the root `README.md`. It does not create
the groups or connect their output to the shared pipelines published by 03F.
Use the supplied CSV inputs for the first shared-pipeline handoff. The mapping CSV assigns each prepared feature to a source group and a storage type. The same identifier and source feature date appear in each group. This notebook ingests the aligned synthetic CSV, reads the completed delivery and retains its extraction provenance.

The Data team owns preparing/stitching source files. This example reads one complete delivery; it does not infer business feature definitions or historical joins between independently arriving tables.


## Select deployed groups and the input CSV

- Read deployment bindings and verify the mapping matches the deployed schema.
- Verify the AWS account and inspect the configured groups and field counts.
- Read the sample while preserving string identifiers and literal category codes.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, sys
import boto3
import pandas as pd
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'mlops-models').is_dir())
sys.path.insert(0, str(ROOT / 'mlops-features'))
from mlops_features.data import ingest_csv, read_dataset
from mlops_features.feature_mapping import load_settings
OUTPUTS = Path(os.environ.get('FEATURE_DEPLOYMENT_OUTPUTS', ROOT / '.runs/features-outputs.json'))
settings = load_settings(OUTPUTS, os.environ.get('FEATURE_MAPPING'))
session = boto3.Session(profile_name=os.environ.get('AWS_PROFILE'), region_name=settings['region'])
assert session.client('sts').get_caller_identity()['Account'] == settings['account']
SOURCE = Path(os.environ.get('FEATURE_SOURCE', ROOT / 'examples/data/curated_fake.csv'))
RUN = ROOT / '.runs/notebook-features' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
RUN.mkdir(parents=True)
CSV_OPTIONS = {'dtype': {settings['record_id']: str}, 'keep_default_na': False, 'na_values': ['', r'\N']}
source_frame = pd.read_csv(SOURCE, **CSV_OPTIONS)
print('Feature groups:', {g['name']: len(g['columns']) + 1 for g in settings['groups'].values()})
print('Results:', RUN)
display(source_frame.head())

## Submit the aligned CSV delivery

- Validate keys and types, then send each group's fields with bounded concurrency.
- Preserve source event times and record completion only after all groups succeed.
- Save the delivery ID, source checksum and upstream revision for repeatable retrieval.


In [ ]:
receipt = ingest_csv(session, settings, SOURCE, upstream_revision='synthetic-example-v1')
(RUN / 'ingestion.json').write_text(json.dumps(receipt, indent=2))
display(receipt)


## Read the complete delivery and retain its provenance

- Query each group at the completed delivery's availability cutoff.
- Require matching ID/date keys across groups and join by those keys.
- Save data.csv and provenance.json at a new extraction location.
- Compare the retrieved values with the source; preserve this receipt if offline visibility times out.


In [ ]:
dataset = read_dataset(session, settings, receipt, RUN / 'dataset')
features = pd.read_csv(RUN / 'dataset/data.csv', **CSV_OPTIONS)
keys = [settings['record_id'], settings['event_time']]
source_frame[settings['event_time']] = pd.to_datetime(source_frame[settings['event_time']], utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
pd.testing.assert_frame_equal(
    source_frame.sort_values(keys).reset_index(drop=True),
    features.sort_values(keys).reset_index(drop=True),
    check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12,
)
display(features.head())
display(dataset)
provenance = json.loads((RUN / 'dataset/provenance.json').read_text())
print('Mapping revision:', provenance['schema']['sha256'])
print('Queries:', [source['query_id'] for source in provenance['sources']])


## Inspect the exported dataset

- Print the exact exported input directory, CSV and provenance locations.
- Use the local dataset directory with Prepare in 03B for a direct Python check.
- Keep the extraction receipt so the selected data can be identified later.

The shared 03F pipelines read `<InputDataRoot>/<ModelId>/`. This notebook's
export is not arranged in that layout, so its `input_uri` must not be passed
directly as `InputDataRoot`. The Feature Store integration increment will add
that handoff; continue using the supplied CSV inputs in 03F for now.


In [ ]:
print('Exported input directory:', dataset['input_uri'])
print('CSV:', dataset['dataset_uri'])
print('Provenance:', dataset['provenance_uri'])
print('Local dataset for 03B:', RUN / 'dataset')
print('Retained dataset receipt:', RUN / 'dataset/provenance.json')
